# Station-MAE — Test Results

Five analyses, all computed from the saved `predictions.pt` dumps (full sliding test set):

Model families that appear here (all share dataset, splits, per-station
normalisation, PFA exclusion, Huber loss and the 13-lead grid — so every
comparison below is apples-to-apples):

| run prefix        | design | forecast produced by |
|-------------------|---|---|
| `v27 ... v29`     | Δ-query cross-attention decoder + station masking | 2-layer decoder |
| `lstm-baseline-*` | per-station recurrence, spatially blind | LSTM + multi-horizon head |

1. **Effect of masking** — mr0.50 vs mr0.00 on the *masked* stations
2. **Model comparison vs persistence** — per-variable, per lead-time (mr0.50)
3. **Terrain** — valley vs mountainous stations
4. **Per-station × variable × season** — which stations work and which don't

Errors are in **physical units** (per-station training σ). Files are ~1.6 GB each, so
§0 aggregates one run at a time and keeps only small summary tables — the raw arrays
are never all in memory at once.

**5. Predicted uncertainty** — for NLL models only (v9). The decoder emits a per-(horizon, station, variable) log σ² alongside the mean, so we can ask whether the model *knows* when it is wrong.


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
import os, sys, glob, datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
# ── Project bootstrap ────────────────────────────────────────────────────────
# This notebook lives in notebooks/ but reads checkpoints/, test_results/ and
# report/ from the project root, and imports data/engine/model from src/.
# Re-running this cell is safe (it only climbs out of notebooks/ once).
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
PROJ = os.getcwd()
if os.path.join(PROJ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, "src"))

RESULTS_ROOT = "test_results"
EXCLUDE      = ["PFA"]
CHUNK        = 2000          # windows processed at a time (memory control)

# Terrain split (§3) — a station is "mountain" if it is high OR steep.
ALT_THRESH   = 1000.0        # m
SLOPE_THRESH = 7.0           # SLOPE_2000M_SIGRATIO1

# Metric shown in the PLOTS below. Tables always report MAE, MSE and RMSE.
#   MAE  = mean |error|      — robust, same units as the variable
#   MSE  = mean error²       — SQUARED units; penalises large errors quadratically
#   RMSE = sqrt(MSE)         — back in the variable's own units
# MSE/RMSE are the ones to quote when large errors matter more than typical
# ones; MAE is the one to quote when they do not. They can rank models
# differently, which is itself worth reporting.
METRIC = "MAE"          # ← "MAE" | "MSE" | "RMSE"

# ── Station x variable exclusions ────────────────────────────────────────────
# Instrument-level exclusions for the results section. Applied ONCE, inside
# aggregate(), by ANDing into the observation-validity mask — so every table and
# plot below (model, persistence, climatology, per-station, seasonal) inherits
# them and none of them can silently disagree with another.
#   "all"  drops the station entirely
#   [list] drops only those variables at that station
DROP_SV = {
    "GES": ["pressure"],
    "PFA": "all",                       # already dropped by EXCLUDE, asserted below
    "LAE": ["wind_u", "wind_v"],        # "wind" = both components
}

# ── Climatology baseline ─────────────────────────────────────────────────────
# clim(T) = the observation at the SAME station and variable exactly 24 h before
# the target time. Forecasting 2023-09-22 14:00…20:00 uses 2023-09-21 14:00…20:00.
# On the 10-min grid that is a fixed 144-step lag. Unlike persistence it carries
# the diurnal cycle, so it is the harder baseline at long lead times: persistence
# decays as the forecast walks away from t0, climatology does not.
CLIM_LAG_STEPS = 144                    # 24 h / 10 min

VARS   = ["temperature", "pressure", "humidity", "wind_u", "wind_v"]
UNITS  = {"temperature":"°C","pressure":"hPa","humidity":"%","wind_u":"m/s","wind_v":"m/s"}
SEASONS = ["DJF","MAM","JJA","SON"]
# One colour per run family. Unknown runs fall back to the cycle below, so a
# new dump never breaks a plot — use COLOR(run) rather than COLORS[run].
COLORS = {
    # transformer line (delta-query decoder, MAE station masking)
    "v15": "#D9663D",          # patch-3 tokens, no residual
    "v17": "#E08A5C",          # token rebalance
    "v19": "#C4502A",          # v17 + MLP value embedding
    "v20": "#2E7D8C",          # + persistence residual head, station_state
    "v27": "#1F5F6B",          # direct head, no masking, no decoder
    "v30-nll":  "#E1A730",     # v27 + heteroscedastic Gaussian NLL head
    "v31":      "#D9663D",     # v27 trained at mask_ratio 0
    "v32-blind":"#5FAF5F",     # no encoder spatial attn + station-local decoder
    # canonical MAE (future in the token grid)
    "simple-mae-v2": "#3F7D3F",
    # encoder-only masked transformer (in-place corruption, pooled readout)
    "masked-tf-v1":  "#7B5EA7",
    # baselines
    "lstm-baseline-v1": "#6A4C93", "lstm-baseline-v2": "#9B7FC0",
    "persistence": "#888888", "climatology": "#CCCCCC",
}
_CYCLE = ["#D9663D", "#2E7D8C", "#5FAF5F", "#7B5EA7", "#C9A227", "#B5651D"]
def COLOR(run):
    """Stable colour for any run name, known or not.

    NOTE: this used to read `return COLOR(run)` inside the `if`, which recursed
    until RecursionError for every KNOWN run — i.e. it failed on exactly the
    runs it had a colour for. Fixed to index the dict.
    """
    if run in COLORS:
        return COLORS[run]
    return _CYCLE[abs(hash(run)) % len(_CYCLE)]

_CAND = [
    os.environ.get("DATA_ROOT", ""),   # set DATA_ROOT to override"/home/renku/work/PeakWeatherDataset",
         os.path.expanduser("~/Documents/ETH/_DAS Project/PeakWeatherDataset"),
         os.path.expanduser("~/PeakWeatherDataset"), "PeakWeatherDataset"]
DATA_ROOT = next((p for p in _CAND if os.path.isdir(p)), _CAND[-1])
plt.rcParams.update({"figure.dpi":110, "font.size":10})

# discover available runs: {run: {mask_ratio: path}}
RUNS = {}
for p in sorted(glob.glob(os.path.join(RESULTS_ROOT, "*", "best_mr*", "predictions.pt"))):
    run = p.split(os.sep)[-3]; mr = p.split(os.sep)[-2].replace("best_", "")
    RUNS.setdefault(run, {})[mr] = p
for r, d in RUNS.items():
    print(f"{r:20s} {sorted(d)}")

In [ ]:
# ── Station metadata: physical scale + terrain class ─────────────────────────
from data.dataset import load_peakweather, StationMAEDataset, compute_obs_stats
ds   = load_peakweather(root=DATA_ROOT)
keep = StationMAEDataset._resolve_keep_indices(ds, EXCLUDE)
STN  = [ds.stations_table.index[i] for i in keep]          # station ids, prediction order
N    = len(STN)

# per-(station, variable) training σ → physical units
_st   = compute_obs_stats(ds, train_years=None, per_station=True)
STD   = np.clip(_st["std"].numpy()[keep][:, :5], 1e-6, None)          # (N, 5)
MEAN  = _st["mean"].numpy()[keep][:, :5]                              # (N, 5)

# terrain
_t     = ds.stations_table.loc[STN]
alt    = _t["station_height"].astype(float).values
slope  = _t["SLOPE_2000M_SIGRATIO1"].astype(float).values
IS_MTN = (alt >= ALT_THRESH) | (slope >= SLOPE_THRESH)                # (N,) bool

# Guard: the station axis of the dumps must match this station list exactly.
# (load_peakweather drops stations lacking the requested parameters, so the raw
#  stations table is larger than the modelled network — never index it directly.)
import torch as _t
_probe = _t.load(next(iter(next(iter(RUNS.values())).values())),
                 map_location="cpu", weights_only=False)["preds"].shape[2]
assert _probe == N, (f"station mismatch: predictions have {_probe} stations but the "
                     f"dataset resolved {N}. Check EXCLUDE={EXCLUDE}.")
del _probe

print(f"stations: {N}   mountain: {IS_MTN.sum()}   valley: {(~IS_MTN).sum()}")
print(f"  mountain: alt {alt[IS_MTN].mean():.0f} m, slope {slope[IS_MTN].mean():.1f}")
print(f"  valley  : alt {alt[~IS_MTN].mean():.0f} m, slope {slope[~IS_MTN].mean():.1f}")

# ── Station x variable exclusion mask ───────────────────────────────────────
# KEEP is (N, 5) and is ANDed into the validity mask in aggregate(). Unknown
# station or variable names raise instead of silently dropping nothing — a
# typo'd abbreviation is otherwise indistinguishable from a clean run.
KEEP, _rep = np.ones((N, len(VARS)), bool), []
for _stn, _spec in DROP_SV.items():
    if _stn not in STN:
        if _stn in EXCLUDE:
            _rep.append(f"  {_stn:<4}  already excluded from the network by EXCLUDE")
            continue
        raise KeyError(f"DROP_SV names station {_stn!r}, which is neither in the "
                       f"modelled network nor in EXCLUDE={EXCLUDE}. Check the "
                       f"abbreviation against ds.stations_table.index.")
    _vs = list(VARS) if _spec == "all" else list(_spec)
    _bad = [v for v in _vs if v not in VARS]
    if _bad:
        raise KeyError(f"DROP_SV[{_stn!r}] names unknown variables {_bad}; VARS={VARS}")
    _si = STN.index(_stn)
    for _v in _vs:
        KEEP[_si, VARS.index(_v)] = False
    _rep.append(f"  {_stn:<4}  station index {_si:3d}   dropped: {', '.join(_vs)}")

print("\nstation x variable exclusions")
print("\n".join(_rep))
print(f"  cells kept: {KEEP.sum()}/{KEEP.size}"
      f"  ({KEEP.size - KEEP.sum()} station-variable pairs dropped)")

### Climatology lookup

`clim(T) = obs(T - 24 h)` at the same station and variable. The dumps do not
contain the previous day, so this reads the raw parquet observations and indexes
them by absolute time. The verification cell below is the load-bearing part: it
denormalises the dump's own targets and checks them against the parquet values at
the same timestamps. If the station ordering, the hour-to-row mapping or the
normalisation statistics are wrong, climatology would be silently misaligned and
every number in section 2 would be wrong — so that check raises rather than warns.

In [ ]:
# ── Climatology lookup: obs(T - 24 h) ───────────────────────────────────────
import pandas as pd
from data.dataset import build_observations, TEST_YEARS

_obs, _msk, _ts = build_observations(ds)          # (T, N_all, 6) PHYSICAL, raw
_ts = pd.DatetimeIndex(_ts)

# Restrict to the test years plus CLIM_LAG_STEPS rows of run-up, so the first
# test timestamp still has a predecessor 24 h earlier. Full history would be
# ~1.5 GB; this slice is ~0.4 GB.
_sel  = np.where(_ts.year.isin(TEST_YEARS))[0]
_lo   = max(0, int(_sel[0]) - CLIM_LAG_STEPS)
_hi   = int(_sel[-1]) + 1
OBS_P = _obs[_lo:_hi][:, keep, :5].numpy()                 # (Tc, N, 5) physical
OBS_M = _msk[_lo:_hi][:, keep, :5].numpy() > 0.5           # (Tc, N, 5) bool
TS_C  = _ts[_lo:_hi]
del _obs, _msk, _sel

# A fixed 144-step lag is 24 h only on a strictly regular grid. Verify, do not assume.
_step = np.unique(np.diff(TS_C.asi8))
assert len(_step) == 1 and _step[0] == 10 * 60 * 10**9, \
    f"observation grid is not regular 10-min: found steps {_step}"
H0   = (TS_C[0] - pd.Timestamp("1970-01-01", tz="UTC")) / pd.Timedelta("1h")
NT_C = len(TS_C)

# Both baselines are DIFFERENCES OF RAW OBSERVATIONS, so they are computed
# directly in physical units. Nothing here is normalised: no MEAN, no STD, no
# round trip. (The model error still uses |p - t| * STD, because the model's
# prediction only exists in normalised space — but MEAN cancels in that
# difference, so it never appears either.)

def base_at(th):
    """
    th (c, K) hours since epoch -> raw physical sources for both baselines.

        truth       obs(t)          the verification target
        pers        obs(t0)         last observation, carried forward
        clim        obs(t - 24 h)   same clock time yesterday

    Returned with each source's own availability mask, since a baseline is only
    defined where ITS source exists.
    """
    ti = np.rint((th - H0) * 6.0).astype(np.int64)     # 6 rows per hour
    pi = ti[:, 0:1]                                    # delta_steps[:,0]==0 -> t0
    ci = ti - CLIM_LAG_STEPS
    okc = (ci >= 0) & (ci < NT_C)
    cg  = np.clip(ci, 0, NT_C - 1)
    return (OBS_P[ti],                                 # truth   (c,K,N,5)
            OBS_P[pi], OBS_M[pi],                      # persistence (broadcasts over K)
            OBS_P[cg], OBS_M[cg] & okc[:, :, None, None])   # climatology

print(f"climatology window: {TS_C[0]}  ..  {TS_C[-1]}   ({NT_C:,} rows)")
print(f"  lag {CLIM_LAG_STEPS} steps = {CLIM_LAG_STEPS*10/60:.0f} h")
print(f"  observation coverage in window: {OBS_M.mean()*100:.1f}%")

# ── Exceedance thresholds for the extremes scores ───────────────────────────
# p95 of the OBSERVED distribution, per station and variable, over the test
# period. Per-station on purpose: "extreme" must mean extreme FOR THAT SITE,
# or the score would just re-measure which stations are windy.
# Wind speed is derived from u,v jointly — p95 on u alone is a strong
# easterly, which is not the same event as a storm.
_ui, _vi = VARS.index("wind_u"), VARS.index("wind_v")
_o = np.where(OBS_M, OBS_P, np.nan)
Q95 = np.nanquantile(_o, 0.95, axis=0)                      # (N, 5)
Q95_SPD = np.nanquantile(np.sqrt(_o[:, :, _ui]**2 + _o[:, :, _vi]**2),
                         0.95, axis=0)                      # (N,)
print(f"p95 thresholds (median over stations): "
      f"temperature {np.nanmedian(Q95[:, 0]):.1f} °C, "
      f"wind speed {np.nanmedian(Q95_SPD):.1f} m/s")
del _o

In [ ]:
# ── Verify station order, time index and normalisation against a real dump ──
import torch
_p  = next(iter(next(iter(RUNS.values())).values()))
_d  = torch.load(_p, map_location="cpu", weights_only=False)
_th = _d["target_hours"][:200].numpy()
_tt = _d["targets"][:200, :, :, :5].numpy()
_tm = _d["masks"][:200, :, :, :5].numpy() > 0.5
_ti = np.rint((_th - H0) * 6.0).astype(np.int64)
assert _ti.min() >= 0 and _ti.max() < NT_C, \
    "dump target times fall outside the climatology slice — TEST_YEARS mismatch?"

_raw  = OBS_P[_ti]                                    # physical, from parquet
_dump = _tt * STD[None, None] + MEAN[None, None]      # physical, from the dump
_ok   = _tm & OBS_M[_ti]
_dev  = np.abs(_raw - _dump)[_ok]
print(f"checked {_ok.sum():,} target values from {os.path.relpath(_p)}")
print(f"  max |parquet - dump| = {_dev.max():.3e}    median = {np.median(_dev):.3e}")
assert _dev.max() < 1e-2, (
    f"targets in the dump do not match the parquet observations at the same "
    f"timestamps (max deviation {_dev.max():.3e}). One of: station ordering, the "
    f"hour->row mapping, or the normalisation statistics is wrong. Climatology "
    f"would be misaligned and section 2 invalid. STOP HERE.")
print("  OK — station order, time index and normalisation all agree.")
del _d, _tt, _tm, _raw, _dump, _ok, _dev

## 0. Aggregation pass

Each dump is streamed in chunks; we accumulate absolute-error sums and counts at the
granularity every later section needs, then release the arrays. **Persistence** is
derived from the same file (`targets[:, 0]` = the last observation carried forward),
so it sits on identical windows and masks.

In [ ]:
# ── Stream each run and accumulate summary tables ────────────────────────────
import torch

SEASONS = ["DJF", "MAM", "JJA", "SON"]
TOD     = ["00-06 UTC", "06-12 UTC", "12-18 UTC", "18-24 UTC"]

def _season_idx(hours):                     # hours since epoch → 0..3
    # vectorised (the old per-element datetime loop ran ~26k times per chunk)
    mth = (pd.Timestamp("1970-01-01", tz="UTC")
           + pd.to_timedelta(np.asarray(hours).ravel(), unit="h")).month.values
    s = np.select([np.isin(mth, [12, 1, 2]), np.isin(mth, [3, 4, 5]),
                   np.isin(mth, [6, 7, 8])], [0, 1, 2], 3)
    return s.reshape(np.shape(hours))

def _tod_idx(hours):                        # hours since epoch → 0..3 (6-h bucket)
    # UTC, deliberately. Civil local time would alias against the 90-min origin
    # stride via DST and load the buckets with different seasonal mixes.
    # Swiss solar time is UTC + ~0:32, so UTC buckets track the sun closely.
    return (np.asarray(hours).astype(np.int64) % 24) // 6

# ── Disk cache ───────────────────────────────────────────────────────────────
# Streaming a 1.6-2 GB dump takes minutes; the accumulators it produces are a
# few hundred kB. Cache them, keyed by everything that changes the RESULT:
#   • the dump's path, mtime and size — a re-dumped predictions.pt invalidates
#     itself automatically, so a stale cache cannot survive a re-run
#   • the exclusions, the climatology lag, the variable list
#   • a signature of STD, so new normalisation statistics invalidate too
#   • AGG_VERSION — bump this by hand whenever aggregate() changes WHAT it
#     computes, or old caches will be served for new code
import hashlib, json as _json
AGG_VERSION = "2026-08-21.c"   # +bias, dispersion, p95 scores, wind_speed
AGG_CACHE   = os.path.join(PROJ, "analysis_outputs", "cache")
os.makedirs(AGG_CACHE, exist_ok=True)

def _agg_key(path):
    st = os.stat(path)
    payload = _json.dumps({
        "version":  AGG_VERSION,
        "dump":     os.path.abspath(path),
        "mtime":    int(st.st_mtime),
        "size":     st.st_size,
        "exclude":  sorted(EXCLUDE),
        "drop_sv":  {k: ("all" if v == "all" else sorted(v))
                     for k, v in sorted(DROP_SV.items())},
        "clim_lag": CLIM_LAG_STEPS,
        "vars":     list(VARS),
        "std_sig":  round(float(np.asarray(STD).sum()), 6),
    }, sort_keys=True)
    return hashlib.sha1(payload.encode()).hexdigest()[:16]

def aggregate(path, use_cache=True):
    """Return dicts of summed |err| and counts at several granularities.

    Results are cached under analysis_outputs/cache/. Pass use_cache=False to
    force a recompute, or delete the directory to rebuild everything.
    """
    _cf = os.path.join(AGG_CACHE, f"agg_{_agg_key(path)}.npz")
    if use_cache and os.path.isfile(_cf):
        _z = np.load(_cf)
        _A = {k[2:]: _z[k] for k in _z.files if k.startswith("A_")}
        _C = {k[2:]: _z[k] for k in _z.files if k.startswith("C_")}
        return _A, _C, _z["grid"]

    d = torch.load(path, map_location="cpu", weights_only=False)
    P, T, M = d["preds"], d["targets"], d["masks"]
    TH, MI  = d["target_hours"], d.get("masked_idx")
    Mw, K   = P.shape[0], P.shape[1]
    # Sigma|e| under each key, and Sigma e^2 under key + "_sq", so MAE, MSE and
    # RMSE all come from one streaming pass over the dump.
    #   _p = persistence  (carry y(t0) forward)
    #   _c = climatology  (the observation 24 h before the target)
    #   _m = masked stations only
    _shapes = {
        "kv":(K,len(VARS)), "kv_p":(K,len(VARS)), "kv_c":(K,len(VARS)),
        "kv_m":(K,len(VARS)), "kv_m_p":(K,len(VARS)), "kv_m_c":(K,len(VARS)),
        "sv":(N,len(VARS)), "svs":(N,len(VARS),4),
        # season x time-of-day x lead x variable — marginalise to get either.
        # 4*4*13*5 = 1040 cells per key, so this is essentially free.
        "stk":(4,4,K,len(VARS)), "stk_p":(4,4,K,len(VARS)),
        "stk_c":(4,4,K,len(VARS)),
        # per (lead, station, variable) — the granularity §4 and the exported
        # table need. 13*155*5 = 10,075 cells per key.
        "knv":(K,N,len(VARS)), "knv_p":(K,N,len(VARS)), "knv_c":(K,N,len(VARS)),
        # signed sums -> bias. |err| and err^2 discard the SIGN: a model always
        # 0.4 degC too warm looks identical to one 0.4 too warm half the time
        # and 0.4 too cold the rest. Only bias separates a correctable offset
        # from irreducible scatter.
        "knv_sgn":(K,N,len(VARS)), "knv_p_sgn":(K,N,len(VARS)),
        "knv_c_sgn":(K,N,len(VARS)),
        # Dispersion: sums of forecast and observation and of their squares,
        # so std(pred)/std(obs) can be formed per cell. A ratio below 1 is the
        # smoothing signature — hedging toward the mean, which MAE rewards.
        "d_f":(K,N,len(VARS)), "d_f2":(K,N,len(VARS)),
        "d_o":(K,N,len(VARS)), "d_o2":(K,N,len(VARS)),
        "d_pf":(K,N,len(VARS)), "d_pf2":(K,N,len(VARS)),
        # p95 contingency: hits / misses / false alarms per cell.
        "x_h":(K,N,len(VARS)), "x_m":(K,N,len(VARS)), "x_f":(K,N,len(VARS)),
        "x_ph":(K,N,len(VARS)), "x_pm":(K,N,len(VARS)), "x_pf":(K,N,len(VARS)),
        # Derived wind SPEED sqrt(u^2+v^2): own metrics, dispersion, scores.
        "w":(K,N), "w_sgn":(K,N), "w_f":(K,N), "w_f2":(K,N),
        "w_o":(K,N), "w_o2":(K,N), "w_h":(K,N), "w_m":(K,N), "w_fa":(K,N),
        "w_p":(K,N), "w_p_sgn":(K,N),
    }
    A = {k: np.zeros(sh) for k, sh in _shapes.items()}
    A.update({k + "_sq": np.zeros(sh) for k, sh in _shapes.items()})
    # Every key gets its OWN counter. Persistence and climatology are valid on
    # strict SUBSETS of the model's valid set (they additionally need y(t0), or
    # the observation 24 h earlier, to exist), so dividing their error sums by
    # the model's count understates their error and flatters the model's skill.
    C = {k: np.zeros(sh) for k, sh in _shapes.items()}

    for a in range(0, Mw, CHUNK):
        b   = min(a+CHUNK, Mw)
        p   = P[a:b].numpy(); t = T[a:b,:,:,:5].numpy()
        # KEEP applies the station x variable exclusions once, here, so every
        # downstream number inherits them.
        m   = (M[a:b,:,:,:5].numpy() > 0.5) & KEEP[None,None,:,:]
        err = np.abs(p - t) * STD[None,None,:,:]                     # (c,K,N,V) physical

        # Baselines, straight off the raw observations — no normalisation.
        th  = TH[a:b].numpy()
        truth, pers, mper, clim, mcl = base_at(th)
        errp = np.abs(pers - truth)                                  # physical, degC/hPa/%/(m/s)
        errc = np.abs(clim - truth)
        mp   = m & mper                                              # valid at Δ0 AND horizon
        mc   = m & mcl                                               # valid now AND 24 h ago
        sea  = _season_idx(th)                                       # (c,K)
        tod  = _tod_idx(th)                                          # (c,K)

        e2, ep2, ec2 = err**2, errp**2, errc**2
        A["kv"]      += (err *m ).sum(axis=(0,2));  C["kv"]   += m.sum(axis=(0,2))
        A["kv_sq"]   += (e2  *m ).sum(axis=(0,2))
        A["kv_p"]    += (errp*mp).sum(axis=(0,2));  C["kv_p"] += mp.sum(axis=(0,2))
        A["kv_p_sq"] += (ep2 *mp).sum(axis=(0,2))
        A["kv_c"]    += (errc*mc).sum(axis=(0,2));  C["kv_c"] += mc.sum(axis=(0,2))
        A["kv_c_sq"] += (ec2 *mc).sum(axis=(0,2))

        # ── season x time-of-day, per lead and variable ──────────────────
        # Both indices are taken from the TARGET time, per (window, lead), so a
        # +6h forecast issued at 20:00 lands in the 00-06 bucket of the NEXT day
        # — which is the regime it is actually predicting.
        _kax = np.broadcast_to(np.arange(K)[None, :, None, None], err.shape)
        _vax = np.broadcast_to(np.arange(len(VARS))[None, None, None, :], err.shape)
        _sax = np.broadcast_to(sea[:, :, None, None], err.shape)
        _tax = np.broadcast_to(tod[:, :, None, None], err.shape)
        for _key, _e, _e2, _mk in (("stk", err, e2, m),
                                   ("stk_p", errp, ep2, mp),
                                   ("stk_c", errc, ec2, mc)):
            _i = (_sax[_mk], _tax[_mk], _kax[_mk], _vax[_mk])
            np.add.at(A[_key],          _i, _e[_mk])
            np.add.at(A[_key + "_sq"],  _i, _e2[_mk])
            np.add.at(C[_key],          _i, 1.0)

        if MI is not None and MI.shape[1] > 0:                       # masked-station subset
            mi  = MI[a:b].numpy()
            sel = np.zeros((b-a, N), bool)
            np.put_along_axis(sel, mi, True, axis=1)
            sm  = m  & sel[:,None,:,None]
            smp = mp & sel[:,None,:,None]
            smc = mc & sel[:,None,:,None]
            A["kv_m"]      += (err *sm ).sum(axis=(0,2)); C["kv_m"]   += sm.sum(axis=(0,2))
            A["kv_m_sq"]   += (e2  *sm ).sum(axis=(0,2))
            A["kv_m_p"]    += (errp*smp).sum(axis=(0,2)); C["kv_m_p"] += smp.sum(axis=(0,2))
            A["kv_m_p_sq"] += (ep2 *smp).sum(axis=(0,2))
            A["kv_m_c"]    += (errc*smc).sum(axis=(0,2)); C["kv_m_c"] += smc.sum(axis=(0,2))
            A["kv_m_c_sq"] += (ec2 *smc).sum(axis=(0,2))

        # ── bias, dispersion, exceedance, wind speed ─────────────────────
        # Forecast in physical units; truth already is. The dump's target mask
        # doubles as observation validity (proved by the round-trip check).
        pph  = p * STD[None,None,:,:] + MEAN[None,None,:,:]
        A["knv_sgn"]   += ((pph  - truth) * m ).sum(axis=0)
        A["knv_p_sgn"] += ((pers - truth) * mp).sum(axis=0)
        A["knv_c_sgn"] += ((clim - truth) * mc).sum(axis=0)
        A["d_f"]  += (pph  * m ).sum(axis=0); A["d_f2"]  += (pph**2  * m ).sum(axis=0)
        A["d_o"]  += (truth* m ).sum(axis=0); A["d_o2"]  += (truth**2* m ).sum(axis=0)
        A["d_pf"] += (pers * mp).sum(axis=0); A["d_pf2"] += (pers**2 * mp).sum(axis=0)

        _ef, _eo = pph >= Q95[None,None,:,:], truth >= Q95[None,None,:,:]
        A["x_h"] += (( _ef &  _eo) & m).sum(axis=0)
        A["x_m"] += ((~_ef &  _eo) & m).sum(axis=0)
        A["x_f"] += (( _ef & ~_eo) & m).sum(axis=0)
        _pe = pers >= Q95[None,None,:,:]
        A["x_ph"] += (( _pe &  _eo) & mp).sum(axis=0)
        A["x_pm"] += ((~_pe &  _eo) & mp).sum(axis=0)
        A["x_pf"] += (( _pe & ~_eo) & mp).sum(axis=0)

        _mw = m[:,:,:,_ui] & m[:,:,:,_vi]
        _sp = np.sqrt(pph[:,:,:,_ui]**2   + pph[:,:,:,_vi]**2)
        _so = np.sqrt(truth[:,:,:,_ui]**2 + truth[:,:,:,_vi]**2)
        A["w"]     += (np.abs(_sp-_so)*_mw).sum(axis=0); C["w"] += _mw.sum(axis=0)
        A["w_sq"]  += ((_sp-_so)**2 * _mw).sum(axis=0)
        A["w_sgn"] += ((_sp-_so)    * _mw).sum(axis=0)
        A["w_f"]   += (_sp*_mw).sum(axis=0); A["w_f2"] += (_sp**2*_mw).sum(axis=0)
        A["w_o"]   += (_so*_mw).sum(axis=0); A["w_o2"] += (_so**2*_mw).sum(axis=0)
        _hf, _ho = _sp >= Q95_SPD[None,None,:], _so >= Q95_SPD[None,None,:]
        A["w_h"]  += (( _hf &  _ho) & _mw).sum(axis=0)
        A["w_m"]  += ((~_hf &  _ho) & _mw).sum(axis=0)
        A["w_fa"] += (( _hf & ~_ho) & _mw).sum(axis=0)
        _mwp = _mw & mp[:,:,:,_ui] & mp[:,:,:,_vi]
        _spp = np.sqrt(pers[:,:,:,_ui]**2 + pers[:,:,:,_vi]**2)
        A["w_p"]     += (np.abs(_spp-_so)*_mwp).sum(axis=0)
        C["w_p"]     += _mwp.sum(axis=0)
        A["w_p_sq"]  += ((_spp-_so)**2 * _mwp).sum(axis=0)
        A["w_p_sgn"] += ((_spp-_so)    * _mwp).sum(axis=0)

        A["knv"]      += (err *m ).sum(axis=0);  C["knv"]   += m.sum(axis=0)
        A["knv_sq"]   += (e2  *m ).sum(axis=0)
        A["knv_p"]    += (errp*mp).sum(axis=0);  C["knv_p"] += mp.sum(axis=0)
        A["knv_p_sq"] += (ep2 *mp).sum(axis=0)
        A["knv_c"]    += (errc*mc).sum(axis=0);  C["knv_c"] += mc.sum(axis=0)
        A["knv_c_sq"] += (ec2 *mc).sum(axis=0)

        A["sv"]    += (err*m).sum(axis=(0,1)); C["sv"] += m.sum(axis=(0,1))
        A["sv_sq"] += (e2 *m).sum(axis=(0,1))
        for si in range(4):                                          # station × season
            w = (sea == si)                                          # (c,K)
            if not w.any(): continue
            ww = w[:,:,None,None]
            A["svs"][:,:,si]    += (err*m*ww).sum(axis=(0,1))
            A["svs_sq"][:,:,si] += (e2 *m*ww).sum(axis=(0,1))
            C["svs"][:,:,si]    += (m*ww).sum(axis=(0,1))
        del p,t,m,err,truth,pers,mper,clim,mcl,errp,errc,mp,mc,sea,tod,e2,ep2,ec2
        del pph,_ef,_eo,_pe,_mw,_sp,_so,_hf,_ho,_mwp,_spp
        del _kax,_vax,_sax,_tax
    grid = d["delta_steps"][0].numpy().astype(int)
    assert grid[0] == 0, (f"persistence assumes lead 0 is the first column, "
                          f"but delta_steps[0] = {grid[0]}")
    np.savez_compressed(_cf, grid=grid,
                        **{f"A_{k}": v for k, v in A.items()},
                        **{f"C_{k}": v for k, v in C.items()})
    del d, P, T, M
    return A, C, grid

AGG = {}
for run, mrs in RUNS.items():
    for mr, path in mrs.items():
        _hit = os.path.isfile(os.path.join(AGG_CACHE, f"agg_{_agg_key(path)}.npz"))
        A, C, GRID = aggregate(path)
        AGG[(run, mr)] = (A, C)
        print(f"  {'cached  ' if _hit else 'streamed'} {run:20s} {mr}")
# Lead labels. The previous formula used int(g)*10//60, which floored 90 min to
# "+1h" — so steps 6 and 9 (1h and 1h30) both printed "+1h", as did 2h/2h30 and
# every other half-hour. Six of the thirteen ticks were duplicates.
def _lead_label(g):
    mins = int(g) * 10
    if mins == 0:   return "t=0"
    if mins < 60:   return f"+{mins}min"
    h, r = divmod(mins, 60)
    return f"+{h}h" if r == 0 else f"+{h}h{r:02d}"
LEAD = [_lead_label(g) for g in GRID]
print("horizons:", LEAD)

# ── Δ=0 is not a forecast ────────────────────────────────────────────────────
# At Δ=0 the target IS the last observation, so for any station the encoder can
# see, the answer is an input. It measures reconstruction, not skill, sits on a
# different scale from the forecast horizons, and compresses the y-axis of any
# lead-time plot it appears in.
#
# Set DROP_DELTA0 = True to drop it from the lead-time plots.
DROP_DELTA0 = False
K0     = 1 if (DROP_DELTA0 and len(GRID) and int(GRID[0]) == 0) else 0
KSL    = slice(K0, None)      # slice the horizon axis of any "kv*" array
LEAD_F = LEAD[K0:]            # matching tick labels
NK     = len(LEAD_F)
print(f"lead-time plots: {'excluding' if K0 else 'including'} Δ=0"
      f"  →  {NK} horizons, {LEAD_F[0]} … {LEAD_F[-1]}")

# ── Metric helpers ───────────────────────────────────────────────────────────
def M(A, C, key, cnt_key=None, m=None):
    """
    Accumulated sums → MAE / MSE / RMSE in physical units.

        MAE  = Σ|e| / n        MSE = Σe² / n        RMSE = √MSE

    `key`     picks the granularity: "kv" (horizon×var), "sv" (station×var),
              "svs" (station×var×season), "kv_m" (masked only), "kv_p"
              (persistence), …
    `cnt_key` overrides the counter. Rarely needed now: "kv_p" and "kv_c" carry
              their own counts, because persistence and climatology are valid on
              subsets of the model's valid set and must not borrow its
              denominator.
    `m`       overrides the global METRIC for one call.
    """
    m = (m or METRIC).upper()
    n = np.maximum(C[cnt_key or key], 1)
    if m == "MAE":
        return A[key] / n
    v = A[key + "_sq"] / n
    return v if m == "MSE" else np.sqrt(v)


def ulab(var, m=None):
    """Unit label — MSE is in SQUARED units, so it must not be mislabelled."""
    m = (m or METRIC).upper()
    return f"{UNITS[var]}²" if m == "MSE" else UNITS[var]


def nrm(var_idx, m=None):
    """
    Divisor that turns a physical metric into a normalised one (per station).
    MAE and RMSE scale with σ; MSE scales with σ².
    """
    m = (m or METRIC).upper()
    s = STD[:, var_idx]
    return s**2 if m == "MSE" else s


print(f"\nMETRIC = {METRIC}   (plots follow this; tables show MAE, MSE and RMSE)")

## 1. Effect of masking — mr0.50 vs mr0.00, on the masked stations

The stations hidden at mr0.50 (`masked_idx`) are evaluated in **both** runs, so the
comparison isolates what the encoder loses by not seeing them. mr0.00 is the same
stations *visible*; mr0.50 is the same stations *inferred from neighbours*.

> **Δ=0 is not comparable across model families.** The k=0 slot means different
> things depending on how the model was supervised:
>
> | family | Δ=0 on MASKED stations | Δ=0 on VISIBLE stations |
> |---|---|---|
> | `v9 … v15`, `simple-mae-*`, `masked-tf-*` | trained — this *is* the gap-filling output | **untrained**: the loss skips visible stations at Δ=0, since a station's own last observation is in its input and supervising it would reward copying |
> | `lstm-baseline-*` | n/a (cannot mask) | trained — all K slots are supervised |
>
> So read Δ=0 only on masked stations, and never compare a masked-model's Δ=0
> against the LSTM's. Everything from Δ=30 min onward is directly comparable.

## 2. Model comparison vs persistence and climatology

Per-variable error against lead time, in whichever metric `METRIC` is set to.

Two reference baselines, and they fail in opposite directions:

- **persistence** — carry `y(t0)` forward. Near-perfect at short lead, degrades
  monotonically as the forecast walks away from the last observation.
- **climatology** — the observation 24 h earlier. Roughly flat in lead time, since
  it carries the diurnal cycle rather than the current state. It is the harder
  reference beyond a few hours, and the one that shows whether a model has learned
  anything more than "tomorrow looks like today".

A model that beats persistence at +6h but not climatology has not learned the
diurnal cycle; the reverse means it tracks the current state but not the daily
rhythm. The table reports skill against both, on MAE and MSE.


In [ ]:
# ── Per-variable error vs lead time: models + persistence ───────────────────
# Plot follows METRIC (set in the config cell); the table below reports skill
# on BOTH MAE and MSE. Skill = 1 − model/persistence, so > 0 beats persistence.
# MSE-skill is the harsher test: it is dominated by the worst forecasts.
# Which mask ratio to show per run.
#
# This previously preferred mr0.50 wherever it existed, which silently put the
# transformers at mr0.50 and the LSTM at mr0.00 (the LSTM has no masking, so it
# only ever dumps mr0.00) — in the SAME comparison plot. Those are different
# tasks: at mr0.50 half the stations are hidden from the encoder. The headline
# comparison must hold the regime fixed.
#
# mr0.00 is the common regime: every run has it, and it is the one the LSTM and
# simple-MAE numbers were produced in.
COMPARE_MR = "mr0.00"           # ← "mr0.00" (comparable) | "mr0.50" (trained setting)

PREF = {}                       # run → mask ratio to display
for r in RUNS:
    PREF[r] = COMPARE_MR if COMPARE_MR in RUNS[r] else sorted(RUNS[r])[0]
_mixed = set(PREF.values())
if len(_mixed) > 1:
    print(f"⚠ runs are NOT all at the same mask ratio: "
          + ", ".join(f"{r}@{m}" for r, m in sorted(PREF.items()))
          + "\n  cross-model differences below are confounded with the regime.")
else:
    print(f"all runs compared at {COMPARE_MR}")

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (vi, v) in zip(axes.ravel(), enumerate(VARS)):
    for run, mr in PREF.items():
        A, C = AGG[(run, mr)]
        ax.plot(range(NK), M(A, C, "kv")[KSL, vi], "o-", ms=3, lw=1.5,
                color=COLOR(run), label=f"{run} ({mr})")
    # baselines — identical across runs, so take the first
    A, C = AGG[list(AGG)[0]]
    ax.plot(range(NK), M(A, C, "kv_p")[KSL, vi],
            "--", color=COLORS["persistence"], lw=1.6, label="persistence")
    ax.plot(range(NK), M(A, C, "kv_c")[KSL, vi],
            ":", color=COLORS["climatology"], lw=2.0, label="climatology (-24h)")
    ax.set_xticks(range(NK))
    ax.set_xticklabels(LEAD_F, rotation=45, ha="right", fontsize=7)
    ax.set_title(f"{v}  [{ulab(v)}]"); ax.grid(alpha=0.3)
# One shared legend in the unused 6th panel. Per-axes legends sat on top of the
# curves in every panel, and at 6 entries they covered the +2h..+4h region —
# exactly where the models separate.
_lax = axes.ravel()[-1]; _lax.axis("off")
_h, _l = axes.ravel()[0].get_legend_handles_labels()
_lax.legend(_h, _l, loc="center", fontsize=9, frameon=False,
            title=f"{METRIC}  ·  mask ratio {COMPARE_MR[2:]}")
fig.suptitle(f"{METRIC} vs lead time, by variable — models vs persistence "
             f"and climatology (-24h)"
             + ("  ·  Δ=0 excluded" if K0 else ""), y=1.02)
plt.tight_layout(); plt.show()

### 2b. The same comparison, split by season and by time of day

The pooled curves above average over conditions that behave very differently.
Two splits, both taken from the **target** time of each (window, lead) pair —
so a +6h forecast issued at 20:00 UTC is counted in the `00-06` bucket of the
next day, the regime it is actually predicting.

* **Season** — DJF / MAM / JJA / SON.
* **Time of day** — 6-hour UTC buckets. **UTC, not civil local time**: the
  evaluation origins sit on a 90-minute grid, so binning by local time aliases
  against DST and loads each bucket with a different seasonal mix (a bug that
  produced a spurious sawtooth earlier in this project). Swiss solar time is
  UTC + ~0:32, so UTC buckets track the sun closely.

What to look for: persistence degrades most where the atmosphere is changing
fastest (daytime, summer), so both baselines and the model-vs-baseline gap
should be strongly condition-dependent. A model whose curve stays flat across
buckets while persistence swings has learned the diurnal/seasonal structure
rather than tracking state.

Counts are printed per panel group — with 4x4 splits some cells thin out, and a
curve built on few samples should not be over-read.

In [ ]:
PREF.items()

In [ ]:
# ── Helper: pooled metric from the (season, tod, lead, var) accumulators ─────
def _stk(A, C, key, vi, seasons=None, tods=None, metric=None):
    """MAE / MSE / RMSE over a chosen slice of seasons and tod buckets."""
    mm  = (metric or METRIC).upper()
    ss  = slice(None) if seasons is None else seasons
    tt  = slice(None) if tods    is None else tods
    num = A[key + ("_sq" if mm in ("MSE", "RMSE") else "")][ss, tt][..., vi]
    den = C[key][ss, tt][..., vi]
    num = num.reshape(-1, num.shape[-1]).sum(0)          # → (K,)
    den = den.reshape(-1, den.shape[-1]).sum(0)
    out = num / np.maximum(den, 1)
    return (np.sqrt(out) if mm == "RMSE" else out), den

# Y-axis scaling for the split figures.
#   "models"    — limits from the model curves only. Best resolution on the
#                 differences that matter; the 24h-lag runs off the top for
#                 humidity/pressure and is clipped (it is ~2.6x the model
#                 there, and squashing five panels to fit it hides everything).
#   "all"       — limits include both baselines. Nothing is clipped, but model
#                 curves are compressed into the lower third.
YLIM_SOURCE = "models"          # "models" | "all"

def _split_figure(kind, ylim_source=None):
    """kind: 'season' | 'tod'. One row per group, one column per variable.

    The y-axis is SHARED DOWN EACH COLUMN, i.e. one scale per variable across
    all groups. Without this every panel auto-scales to its own data and the
    seasons cannot be compared — a flat-looking JJA panel and a flat-looking
    DJF panel may be an octave apart.
    """
    src = ylim_source or YLIM_SOURCE
    labels = SEASONS if kind == "season" else TOD
    fig, axes = plt.subplots(len(labels), len(VARS),
                             figsize=(4.0 * len(VARS), 2.9 * len(labels)),
                             sharex=True, sharey="col")
    _lim = {vi: [np.inf, -np.inf] for vi in range(len(VARS))}
    Ab, Cb = AGG[list(AGG)[0]]                            # baselines: any run
    for gi, gname in enumerate(labels):
        sel = dict(seasons=[gi]) if kind == "season" else dict(tods=[gi])
        for vi, v in enumerate(VARS):
            ax = axes[gi, vi]
            for run, mr in PREF.items():
                A, C = AGG[(run, mr)]
                y, n = _stk(A, C, "stk", vi, **sel)
                ax.plot(range(NK), y[KSL], "o-", ms=2.5, lw=1.3,
                        color=COLOR(run), label=f"{run}")
                _f = np.isfinite(y[KSL])
                if _f.any():
                    _lim[vi][0] = min(_lim[vi][0], np.nanmin(y[KSL][_f]))
                    _lim[vi][1] = max(_lim[vi][1], np.nanmax(y[KSL][_f]))
            yp, _ = _stk(Ab, Cb, "stk_p", vi, **sel)
            yc, nc = _stk(Ab, Cb, "stk_c", vi, **sel)
            if src == "all":
                for _y in (yp[KSL], yc[KSL]):
                    _f = np.isfinite(_y)
                    if _f.any():
                        _lim[vi][0] = min(_lim[vi][0], np.nanmin(_y[_f]))
                        _lim[vi][1] = max(_lim[vi][1], np.nanmax(_y[_f]))
            else:
                _f = np.isfinite(yp[KSL])          # persistence always fits
                if _f.any():
                    _lim[vi][1] = max(_lim[vi][1], np.nanmax(yp[KSL][_f]))
            ax.plot(range(NK), yp[KSL], "--", lw=1.5,
                    color=COLORS["persistence"], label="persistence")
            ax.plot(range(NK), yc[KSL], ":", lw=1.9,
                    color=COLORS["climatology"], label="climatology (-24h)")
            ax.grid(alpha=0.3)
            ax.set_xticks(range(0, NK, 2))
            ax.set_xticklabels([LEAD_F[i] for i in range(0, NK, 2)],
                               rotation=45, ha="right", fontsize=6.5)
            if gi == 0:
                ax.set_title(f"{v}  [{ulab(v)}]", fontsize=10)
            if vi == 0:
                ax.set_ylabel(f"{gname}\n{METRIC}", fontsize=9)
        print(f"  {gname:10s} n = {int(nc.sum()):>12,} baseline-valid predictions")

    # One scale per variable, applied to every row: panels in a column are now
    # directly comparable. 5% headroom so markers are not clipped by the frame.
    for vi in range(len(VARS)):
        lo, hi = _lim[vi]
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            continue
        pad = 0.05 * (hi - lo)
        for gi in range(len(labels)):
            axes[gi, vi].set_ylim(lo - pad, hi + pad)
        axes[0, vi].set_title(f"{axes[0, vi].get_title()}", fontsize=10)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), fontsize=8.5,
               frameon=False, bbox_to_anchor=(0.5, -0.015))
    ttl = "season" if kind == "season" else "time of day (UTC, target time)"
    _clip = " · y-scale per variable, from model curves (24h-lag may be clipped)" \
            if src == "models" else " · y-scale per variable, all series"
    fig.suptitle(f"{METRIC} vs lead time by variable and {ttl} — "
                 f"models vs persistence and climatology"
                 + ("  ·  Δ=0 excluded" if K0 else "") + _clip, y=1.005)
    plt.tight_layout(); plt.show()
    return fig

print("samples per season:")
_ = _split_figure("season")

In [ ]:
AGG.keys()

In [ ]:
print("samples per time-of-day bucket:")
_ = _split_figure("tod")

In [ ]:
# ── Masked vs non-masked stations, mr0.00 vs mr0.50, single model ────────────
run = ["v27", "v30-nll"]
A5, C5 = AGG[(run[0], "mr0.50")]
A0, C0 = AGG[(run[1], "mr0.50")]

fig, axes = plt.subplots(1, len(VARS), figsize=(4*len(VARS), 3.2), squeeze=False)
axes = axes[0]

for vi, v in enumerate(VARS):
    ax = axes[vi]

    # masked-subset stations: hidden under mr0.50 vs the same stations unmasked under mr0.00
    hid_mr50 = M(A5, C5, "kv_m")[KSL, vi]
    hid_mr00 = (M(A0, C0, "kv_m")[KSL, vi] if C0["kv_m"][:, vi].sum()
                else M(A0, C0, "kv")[KSL, vi])

    # non-masked stations: visible under both mr0.00 and mr0.50
    vis_mr50 = (M(A5, C5, "kv_nm")[KSL, vi] if "kv_nm" in C5 and C5["kv_nm"][:, vi].sum()
                else M(A5, C5, "kv")[KSL, vi])
    vis_mr00 = (M(A0, C0, "kv_nm")[KSL, vi] if "kv_nm" in C0 and C0["kv_nm"][:, vi].sum()
                else M(A0, C0, "kv")[KSL, vi])

    ax.plot(range(NK), hid_mr00, "o-",  color="#2E7D8C", lw=1.5, ms=3, label="v30-nll masked")
    ax.plot(range(NK), hid_mr50, "s-",  color="#D9663D", lw=1.5, ms=3, label="v27 masked")
    ax.plot(range(NK), vis_mr00, "^--", color="#6BA368", lw=1.5, ms=3, label="v30-nll non-masked")
    ax.plot(range(NK), vis_mr50, "d--", color="#8E6BAF", lw=1.5, ms=3, label="v27 non-masked")

    ax.set_xticks(range(NK)); ax.set_xticklabels(LEAD_F, rotation=45, ha="right", fontsize=6)
    ax.set_title(f"{run} · {v} [{ulab(v)}]", fontsize=9); ax.grid(alpha=0.3)
    if vi == 0:
        ax.legend(fontsize=7)

fig.suptitle(f"{METRIC} vs lead time — masked vs non-masked stations, mr0.50 ({run})"
             + ("  ·  Δ=0 excluded" if K0 else ""), y=1.02)
plt.tight_layout(); plt.show()

## 3. Terrain — valley vs mountainous stations

Stations are split by altitude and slope (`ALT_THRESH`, `SLOPE_THRESH` at the top).
Complex terrain is where spatial interpolation is hardest, so this is the clearest test
of whether the models cope with the Alps.

In [ ]:
# ── Error by terrain class, per variable and model ───────────────────────────
rows = {}
for run, mr in PREF.items():
    A, C = AGG[(run, mr)]
    for vi, v in enumerate(VARS):
        for name, sel in [("valley", ~IS_MTN), ("mountain", IS_MTN)]:
            n   = max(C["sv"][sel, vi].sum(), 1)
            mae = A["sv"][sel, vi].sum() / n
            mse = A["sv_sq"][sel, vi].sum() / n
            rows[(f"{run} ({mr})", v, name)] = {
                "MAE": mae, "MSE": mse, "RMSE": np.sqrt(mse),
                "n_stations": int(sel.sum())}
t = pd.DataFrame(rows).T; t.index.names = ["model","variable","terrain"]

for m in ("MAE", "MSE"):
    piv = t[m].unstack("terrain").astype(float)
    piv["ratio mtn/valley"] = piv["mountain"] / piv["valley"]
    display(piv.style.format("{:.3f}")
            .background_gradient(cmap="OrRd", subset=["ratio mtn/valley"])
            .set_caption(f"{m} by terrain — ratio > 1 means the mountains are harder"
                         + ("  (squared units)" if m == "MSE" else "")))

x = np.arange(len(VARS)); nb_ = len(PREF); w = 0.8/max(nb_*2, 1)
fig, ax = plt.subplots(figsize=(11, 4.5))
for i, (run, mr) in enumerate(PREF.items()):
    A, C = AGG[(run, mr)]
    val, mtn = [], []
    for vi in range(len(VARS)):
        for sel, out in ((~IS_MTN, val), (IS_MTN, mtn)):
            n = max(C["sv"][sel, vi].sum(), 1)
            if METRIC == "MAE":   out.append(A["sv"][sel, vi].sum() / n)
            elif METRIC == "MSE": out.append(A["sv_sq"][sel, vi].sum() / n)
            else:                 out.append(np.sqrt(A["sv_sq"][sel, vi].sum() / n))
    ax.bar(x+(2*i-nb_+0.5)*w, val, w, color=COLOR(run), alpha=0.55,
           label=f"{run} valley")
    ax.bar(x+(2*i-nb_+1.5)*w, mtn, w, color=COLOR(run),
           label=f"{run} mountain")
ax.set_xticks(x); ax.set_xticklabels([f"{v}\n[{ulab(v)}]" for v in VARS])
ax.set_ylabel(f"{METRIC} (physical)")
ax.set_title(f"Valley vs mountain {METRIC} by variable")
ax.legend(fontsize=8, ncol=2); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Per-station performance by variable and season

Which stations the model handles well, and which it does not. Errors are normalised by
each station's own training σ (**nMAE**) so stations with different natural variability
are comparable — a raw MAE would simply rank the windiest sites worst.

In [ ]:
# ── Per-station normalised error by variable, best / worst ──────────────────
# Normalised by the station's own training σ so stations are comparable:
#   nMAE, nRMSE  divide by σ      nMSE  divides by σ²   (units must match)
# Prefer the newest transformer run, then anything available.
_PREF_ORDER = ["v20", "v22", "v19", "v17", "v15", "simple-mae-v2",
               "masked-tf-v1", "simple-mae-v1",
               "lstm-baseline-v2", "lstm-baseline-v1", "v9"]
RUN_INSPECT = next((r for r in _PREF_ORDER if r in RUNS), list(RUNS)[0])
MR = PREF[RUN_INSPECT]
A, C = AGG[(RUN_INSPECT, MR)]

_n = np.maximum(C["sv"], 1)
_phys = {"MAE": A["sv"]/_n, "MSE": A["sv_sq"]/_n, "RMSE": np.sqrt(A["sv_sq"]/_n)}
_norm = {"MAE": _phys["MAE"]/STD, "MSE": _phys["MSE"]/STD**2, "RMSE": _phys["RMSE"]/STD}

# ── Absent sensors are NOT perfect forecasts ────────────────────────────────
# A (station, variable) pair with no valid observation divides 0 by
# max(count, 1) and lands at exactly 0.0 — which a row-mean then reads as a
# flawless station. It is not a rounding curiosity: 39 of the 775 pairs here
# are empty, and because normalised error differs ~4x between variables
# (wind ~0.40 vs temperature ~0.09), the five stations with no wind sensor
# took the top five "best" slots outright before this mask existed.
# Mask them to NaN, average only over what a station actually measures, and
# show n_vars so partial stations are visible rather than silently mixed in.
_HAVE = C["sv"] > 100                                    # (N, V)
nrm_tbl = pd.DataFrame(np.where(_HAVE, _norm[METRIC], np.nan),
                       index=STN, columns=VARS)
nrm_tbl["n_vars"] = _HAVE.sum(1)
nrm_tbl["mean"] = nrm_tbl[VARS].mean(axis=1)             # skipna → own sensors
# Ranked on COMPLETE stations only, so the comparison is like-for-like.
_full = nrm_tbl["n_vars"] == len(VARS)
print(f"complete stations: {int(_full.sum())} of {len(STN)}   "
      f"(empty station x variable pairs: {int((~_HAVE).sum())})")
_incomplete = nrm_tbl.index[~_full]
if len(_incomplete):
    print("  partial (ranked separately):",
          ", ".join(f"{s}[{int(nrm_tbl.loc[s, 'n_vars'])}]" for s in _incomplete))
nrm_tbl = nrm_tbl[_full]
nrm_tbl["terrain"] = np.where(IS_MTN, "mountain", "valley")[_full.values]

print(f"{RUN_INSPECT} ({MR}) — per-station n{METRIC} "
      f"(error ÷ station σ{'²' if METRIC=='MSE' else ''})")
display(nrm_tbl.nsmallest(10, "mean").style.format({c_:"{:.3f}" for c_ in VARS+["mean"]})
        .set_caption(f"10 BEST stations (n{METRIC})"))
display(nrm_tbl.nlargest(10, "mean").style.format({c_:"{:.3f}" for c_ in VARS+["mean"]})
        .set_caption(f"10 WORST stations (n{METRIC})"))

# Do MAE and MSE agree on who the worst stations are? If not, those stations
# are hurt by occasional large errors rather than consistent mediocrity.
_wm = pd.DataFrame(np.where(_HAVE, _norm["MAE"], np.nan), index=STN,
                   columns=VARS)[_full.values].mean(axis=1).nlargest(15).index
_ws = pd.DataFrame(np.where(_HAVE, _norm["MSE"], np.nan), index=STN,
                   columns=VARS)[_full.values].mean(axis=1).nlargest(15).index
_only_mse = [s for s in _ws if s not in _wm]
print(f"\nWorst-15 by nMAE vs nMSE: {len(set(_wm) & set(_ws))}/15 in common.")
if _only_mse:
    print(f"  Bad on MSE only (large-error driven): {', '.join(_only_mse)}")

### 4b. Per-station x lead-time metrics, cached to disk

`aggregate()` now also accumulates at **(lead, station, variable)** and caches
every run's accumulators under `analysis_outputs/cache/`. The cache key covers
the dump's path, mtime and size, the exclusions, the climatology lag and a
signature of the normalisation statistics — so a re-dumped `predictions.pt`
invalidates itself and a stale cache cannot be served silently. Bump
`AGG_VERSION` by hand if you change *what* `aggregate()` computes.

The cell below writes a tidy table — one row per
(model, mask ratio, estimator, lead, station, variable) with MAE, RMSE and the
sample count — and reloads it instead of rebuilding whenever it is newer than
every dump. Pairs with no observations are dropped rather than written as 0.

In [ ]:
# ── Shared table: MAE by lead time x station x variable ─────────────────────
# ONE canonical file, identical schema on both sides, so this notebook and
# notebooks/analysis/*.ipynb read and write the same thing:
#     analysis_outputs/tables/per_station_lead_metrics.csv
# Physical units. Rebuilt only when missing or older than the newest dump.
PSL_FILE = os.path.join(PROJ, "analysis_outputs", "tables",
                        "per_station_lead_metrics.csv")
PSL_COLUMNS = ["model", "mask_ratio", "estimator", "delta_steps", "lead",
               "station", "variable", "MAE", "RMSE", "bias", "disp_ratio",
               "POD", "FAR", "CSI", "freq_bias", "n", "source"]
os.makedirs(os.path.dirname(PSL_FILE), exist_ok=True)

_newest_dump = max(os.path.getmtime(p_) for d_ in RUNS.values()
                   for p_ in d_.values())
_fresh = (os.path.isfile(PSL_FILE)
          and os.path.getmtime(PSL_FILE) > _newest_dump)
if _fresh:
    PSL = pd.read_csv(PSL_FILE)
    _missing = [c for c in PSL_COLUMNS if c not in PSL.columns]
    if _missing:
        print(f"  file lacks {_missing} — rebuilding"); _fresh = False
if _fresh:
    print(f"loaded {len(PSL):,} rows from {os.path.relpath(PSL_FILE, PROJ)} "
          f"(newer than every dump)")
else:
    _EST = {"knv": "model", "knv_p": "persistence", "knv_c": "climatology"}
    # dispersion + contingency exist for model and persistence only; the
    # 24h-lag needs no dispersion diagnostic (it IS an observation, so its
    # ratio is ~1 by construction and carries no information about smoothing).
    _DSP = {"knv": ("d_f", "d_f2"), "knv_p": ("d_pf", "d_pf2")}
    _CON = {"knv": ("x_h", "x_m", "x_f"), "knv_p": ("x_ph", "x_pm", "x_pf")}
    frames = []
    for (run, mr), (A, C) in AGG.items():
        for est, tag in _EST.items():
            n = C[est]                                       # (K, N, V)
            with np.errstate(invalid="ignore", divide="ignore"):
                mae  = np.where(n > 0, A[est] / np.maximum(n, 1), np.nan)
                rmse = np.where(n > 0, np.sqrt(A[est + "_sq"]
                                               / np.maximum(n, 1)), np.nan)
                bias = np.where(n > 0, A[est + "_sgn"]
                                / np.maximum(n, 1), np.nan)
                nn = np.maximum(n, 1)
                if est in _DSP:
                    f1, f2 = _DSP[est]
                    vf = A[f2]/nn - (A[f1]/nn)**2
                    vo = A["d_o2"]/np.maximum(C["knv"],1) \
                         - (A["d_o"]/np.maximum(C["knv"],1))**2
                    disp = np.where((n > 0) & (vo > 1e-12),
                                    np.sqrt(np.clip(vf, 0, None))
                                    / np.sqrt(np.clip(vo, 1e-12, None)), np.nan)
                    h, m_, f_ = (A[k] for k in _CON[est])
                    pod  = np.where(h + m_ > 0, h / np.maximum(h + m_, 1), np.nan)
                    far  = np.where(h + f_ > 0, f_ / np.maximum(h + f_, 1), np.nan)
                    csi  = np.where(h + m_ + f_ > 0,
                                    h / np.maximum(h + m_ + f_, 1), np.nan)
                    fbi  = np.where(h + m_ > 0,
                                    (h + f_) / np.maximum(h + m_, 1), np.nan)
                else:
                    disp = pod = far = csi = fbi = np.full_like(mae, np.nan)
            K_, N_, V_ = n.shape
            frames.append(pd.DataFrame({
                "model": run, "mask_ratio": mr, "estimator": tag,
                "delta_steps": np.repeat(GRID, N_ * V_),
                "lead": np.repeat(LEAD, N_ * V_),
                "station": np.tile(np.repeat(list(STN), V_), K_),
                "variable": np.tile(VARS, K_ * N_),
                "MAE": mae.ravel(), "RMSE": rmse.ravel(),
                "bias": bias.ravel(), "disp_ratio": disp.ravel(),
                "POD": pod.ravel(), "FAR": far.ravel(),
                "CSI": csi.ravel(), "freq_bias": fbi.ravel(),
                "n": n.ravel().astype(np.int64),
                "source": "test_results_exploration",
            }))
        # ── derived wind_speed rows (one per lead x station) ─────────────
        for est, tag, cnt_k in (("w", "model", "w"), ("w_p", "persistence", "w_p")):
            nw = C[cnt_k]                                    # (K, N)
            with np.errstate(invalid="ignore", divide="ignore"):
                nn = np.maximum(nw, 1)
                wm = np.where(nw > 0, A[est] / nn, np.nan)
                wr = np.where(nw > 0, np.sqrt(A[est + "_sq"] / nn), np.nan)
                wb = np.where(nw > 0, A[est + "_sgn"] / nn, np.nan)
                if est == "w":
                    vf = A["w_f2"]/nn - (A["w_f"]/nn)**2
                    vo = A["w_o2"]/nn - (A["w_o"]/nn)**2
                    wd = np.where((nw > 0) & (vo > 1e-12),
                                  np.sqrt(np.clip(vf, 0, None))
                                  / np.sqrt(np.clip(vo, 1e-12, None)), np.nan)
                    h, m_, f_ = A["w_h"], A["w_m"], A["w_fa"]
                    wp = np.where(h+m_ > 0, h/np.maximum(h+m_,1), np.nan)
                    wf = np.where(h+f_ > 0, f_/np.maximum(h+f_,1), np.nan)
                    wc = np.where(h+m_+f_ > 0, h/np.maximum(h+m_+f_,1), np.nan)
                    wq = np.where(h+m_ > 0, (h+f_)/np.maximum(h+m_,1), np.nan)
                else:
                    wd = wp = wf = wc = wq = np.full_like(wm, np.nan)
            K2, N2 = nw.shape
            frames.append(pd.DataFrame({
                "model": run, "mask_ratio": mr, "estimator": tag,
                "delta_steps": np.repeat(GRID, N2),
                "lead": np.repeat(LEAD, N2),
                "station": np.tile(list(STN), K2),
                "variable": "wind_speed",
                "MAE": wm.ravel(), "RMSE": wr.ravel(), "bias": wb.ravel(),
                "disp_ratio": wd.ravel(), "POD": wp.ravel(), "FAR": wf.ravel(),
                "CSI": wc.ravel(), "freq_bias": wq.ravel(),
                "n": nw.ravel().astype(np.int64),
                "source": "test_results_exploration"}))

    PSL = pd.concat(frames, ignore_index=True)
    # n == 0 means NO DATA. Dropped, never written as 0.0 — a zero there reads
    # as a perfect forecast and silently wins any "best station" ranking.
    PSL = PSL[PSL["n"] > 0][PSL_COLUMNS]
    PSL.to_csv(PSL_FILE, index=False)
    print(f"wrote {len(PSL):,} rows -> {os.path.relpath(PSL_FILE, PROJ)}")

def psl_curve(model, variable, mask_ratio="mr0.00", estimator="model",
              stations=None, metric="MAE", df=None):
    """Pooled metric vs lead from the shared table.

    COUNT-WEIGHTED, never a mean of per-station means — stations contribute in
    proportion to their valid observations, matching every other number here.
    Mirrors common.psl_curve() in the analysis suite.
    """
    q = (df if df is not None else PSL)
    q = q[(q.model == model) & (q.mask_ratio == mask_ratio)
          & (q.estimator == estimator) & (q.variable == variable)]
    if stations is not None:
        q = q[q.station.isin(list(stations))]
    if metric.upper() == "RMSE":
        num = (q.RMSE ** 2 * q.n).groupby(q.delta_steps).sum()
    else:
        num = (q.MAE * q.n).groupby(q.delta_steps).sum()
    den = q.n.groupby(q.delta_steps).sum().clip(lower=1)
    out = (np.sqrt(num / den) if metric.upper() == "RMSE" else num / den)
    out = out.sort_index()
    return out.index.to_numpy(), out.to_numpy()

print(f"\nmodels x mask ratios: "
      f"{PSL.groupby(['model','mask_ratio']).ngroups}   "
      f"stations: {PSL.station.nunique()}   leads: {PSL.delta_steps.nunique()}")
display(PSL.head(6))

In [ ]:
# ── Plots straight from the shared table (no dumps, no AGG needed) ──────────
# Everything below reads PSL only, so these cells also run in a fresh kernel
# after `PSL = pd.read_csv(PSL_FILE)` — useful for iterating on figures.
_models = [m for m in PSL.model.unique()
           if "mr0.00" in set(PSL[PSL.model == m].mask_ratio)]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (vi, v) in zip(axes.ravel(), enumerate(VARS)):
    for run in _models:
        x, y = psl_curve(run, v)
        ax.plot(range(len(x)), y, "o-", ms=3, lw=1.5,
                color=COLOR(run), label=run)
    for est, style, col, lab in (("persistence", "--", COLORS["persistence"],
                                  "persistence"),
                                 ("climatology", ":", COLORS["climatology"],
                                  "climatology (-24h)")):
        x, y = psl_curve(_models[0], v, estimator=est)
        ax.plot(range(len(x)), y, style, lw=1.7, color=col, label=lab)
    ax.set_xticks(range(len(LEAD)))
    ax.set_xticklabels(LEAD, rotation=45, ha="right", fontsize=7)
    ax.set_title(f"{v}  [{ulab(v)}]"); ax.grid(alpha=.3)
_lax = axes.ravel()[-1]; _lax.axis("off")
_h, _l = axes.ravel()[0].get_legend_handles_labels()
_lax.legend(_h, _l, loc="center", fontsize=9, frameon=False,
            title="MAE [physical units]\nfrom per_station_lead_metrics.csv")
fig.suptitle("MAE vs lead time — rebuilt from the shared per-station table",
             y=1.02)
plt.tight_layout(); plt.show()

# ── Station subsets: the table makes this a one-liner ───────────────────────
# Same curves restricted to terrain classes — impossible before, because the
# old accumulators had no joint (lead x station) granularity.
_mtn = [s for s, m_ in zip(STN, IS_MTN) if m_]
_val = [s for s, m_ in zip(STN, IS_MTN) if not m_]
fig, axes = plt.subplots(1, 5, figsize=(17, 3.3))
for ax, (vi, v) in zip(axes, enumerate(VARS)):
    for subset, name, col in ((_val, "valley", "#1F5F6B"),
                              (_mtn, "mountain", "#C4502A")):
        x, y = psl_curve("v27", v, stations=subset)
        ax.plot(range(len(x)), y, "o-", ms=3, color=col, label=name)
    ax.set_xticks(range(0, len(LEAD), 2))
    ax.set_xticklabels(LEAD[::2], rotation=45, ha="right", fontsize=6.5)
    ax.set_title(f"{v}  [{ulab(v)}]", fontsize=10); ax.grid(alpha=.3)
axes[0].set_ylabel("v27 MAE"); axes[-1].legend(fontsize=8)
fig.suptitle("v27 MAE vs lead by terrain class — from the shared table", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
# ── Dispersion ratio and p95 extremes scores, from the shared table ─────────
def _pool(df, col, model, variable, weight="n", mask_ratio="mr0.00",
          estimator="model"):
    q = df[(df.model==model) & (df.mask_ratio==mask_ratio)
           & (df.estimator==estimator) & (df.variable==variable)
           & df[col].notna()]
    num = (q[col]*q[weight]).groupby(q.delta_steps).sum()
    den = q[weight].groupby(q.delta_steps).sum().clip(lower=1)
    return (num/den).sort_index()

_mods = [m for m in PSL.model.unique()
         if "mr0.00" in set(PSL[PSL.model==m].mask_ratio)]
_VP = VARS + ["wind_speed"]

fig, axes = plt.subplots(1, len(_VP), figsize=(3.2*len(_VP), 3.4), sharey=True)
for ax, v in zip(axes, _VP):
    for run in _mods:
        d = _pool(PSL, "disp_ratio", run, v)
        ax.plot(range(len(d)), d.values, "o-", ms=3, lw=1.3,
                color=COLOR(run), label=run)
    d = _pool(PSL, "disp_ratio", _mods[0], v, estimator="persistence")
    if len(d):
        ax.plot(range(len(d)), d.values, "--", lw=1.4,
                color=COLORS["persistence"], label="persistence")
    ax.axhline(1.0, color="k", lw=1)
    ax.set_title(v, fontsize=10); ax.grid(alpha=.3)
    ax.set_xticks(range(0, len(LEAD), 3))
    ax.set_xticklabels(LEAD[::3], rotation=45, ha="right", fontsize=6.5)
axes[0].set_ylabel("std(pred) / std(obs)"); axes[-1].legend(fontsize=6.5)
fig.suptitle("Dispersion ratio vs lead — below 1 = the model is smoothing "
             "(persistence sits at ~1 by construction)", y=1.05)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (col, lab, ideal) in zip(axes, [
        ("POD", "hit rate (POD) — higher better", 1.0),
        ("FAR", "false-alarm ratio — lower better", 0.0),
        ("CSI", "critical success index — higher better", 1.0),
        ("freq_bias", "frequency bias — 1 = right event count", 1.0)]):
    for run in _mods:
        d = _pool(PSL, col, run, "wind_speed")
        ax.plot(range(len(d)), d.values, "o-", ms=3, lw=1.3,
                color=COLOR(run), label=run)
    d = _pool(PSL, col, _mods[0], "wind_speed", estimator="persistence")
    if len(d):
        ax.plot(range(len(d)), d.values, "--", lw=1.4,
                color=COLORS["persistence"], label="persistence")
    ax.axhline(ideal, color="k", lw=.9, ls=":")
    ax.set_title(lab, fontsize=9.5); ax.grid(alpha=.3)
    ax.set_xticks(range(0, len(LEAD), 3))
    ax.set_xticklabels(LEAD[::3], rotation=45, ha="right", fontsize=6.5)
axes[-1].legend(fontsize=7)
fig.suptitle("Wind-speed extremes: p95 exceedance scores vs lead time "
             "(per-station thresholds)", y=1.04)
plt.tight_layout(); plt.show()

_t = pd.DataFrame({
    run: {c: float(_pool(PSL, c, run, "wind_speed").iloc[4])
          for c in ("disp_ratio", "POD", "FAR", "CSI", "freq_bias")}
    for run in _mods}).T
display(_t.round(3).style.set_caption(
    "wind speed @ +2h — dispersion and p95 exceedance scores"))

In [ ]:
# ── Station × season heatmap for one variable ────────────────────────────────
SEASON_VAR = "temperature"      # ← change variable
vi = VARS.index(SEASON_VAR)
_ns_n = np.maximum(C["svs"][:,vi,:], 1)
if METRIC == "MAE":
    ns = (A["svs"][:,vi,:] / _ns_n) / STD[:,vi:vi+1]
elif METRIC == "MSE":
    ns = (A["svs_sq"][:,vi,:] / _ns_n) / (STD[:,vi:vi+1]**2)
else:
    ns = np.sqrt(A["svs_sq"][:,vi,:] / _ns_n) / STD[:,vi:vi+1]   # (N,4) normalised
order = np.argsort(np.nanmean(ns, axis=1))

fig, ax = plt.subplots(figsize=(7, 9))
im = ax.imshow(ns[order], aspect="auto", cmap="YlOrRd",
               vmin=np.nanpercentile(ns,5), vmax=np.nanpercentile(ns,95))
ax.set_xticks(range(4)); ax.set_xticklabels(SEASONS)
ax.set_yticks([]); ax.set_ylabel(f"stations (n={N}, sorted best → worst)")
ax.set_title(f"{RUN_INSPECT} ({MR}) · {SEASON_VAR} — n{METRIC} by station and season")
plt.colorbar(im, ax=ax, label=f"n{METRIC}", shrink=0.7)
plt.tight_layout(); plt.show()

seas_tbl = pd.DataFrame(ns, index=STN, columns=SEASONS)
seas_tbl["worst season"] = seas_tbl[SEASONS].idxmax(axis=1)
seas_tbl["terrain"] = np.where(IS_MTN, "mountain", "valley")
display(seas_tbl.groupby("terrain")[SEASONS].mean().style.format("{:.3f}")
        .set_caption(f"{SEASON_VAR} n{METRIC} by season and terrain"))
display(seas_tbl.nlargest(12, SEASONS).style.format({s:"{:.3f}" for s in SEASONS})
        .set_caption(f"Stations struggling most on {SEASON_VAR}"))

## 5. Predicted uncertainty vs lead time — v30-nll

The NLL model emits a per-(lead, station, variable) `log_var` alongside the
mean, so it states how uncertain it is about every prediction. This section
plots that predicted spread against lead time, per variable.

**Note on naming:** the variance head is on **v30-nll**. `v31` is the
mask-ratio-0 Huber run and carries no `log_var`; v30-nll is the only dump with
predicted uncertainty, and it has it at both mr0.00 and mr0.50.

**Units.** `log_var` is in normalised space, so
`sigma_phys = exp(0.5 * log_var) * STD` per station and variable. Plotting
sigma rather than variance keeps the y-axis in °C / hPa / % / (m/s), directly
comparable to RMSE. Variance is simply the square.

**Why RMSE is overlaid.** For a calibrated Gaussian `E[e^2] = sigma^2`, so the
predicted RMSE `sqrt(mean(sigma^2))` should equal the realised RMSE. Their
ratio is the calibration factor: `>1` overconfident (real error exceeds the
claimed spread), `<1` underconfident. Everything is pooled over stations and
windows with the same station x variable exclusions used elsewhere.

In [ ]:
# ── Stream v30-nll and accumulate predicted variance + realised error ────────
import torch
NLL_RUN = "v30-nll"
assert NLL_RUN in RUNS, f"{NLL_RUN} not found; available: {sorted(RUNS)}"

SIG = {}
for mr, path in sorted(RUNS[NLL_RUN].items()):
    d = torch.load(path, map_location="cpu", weights_only=False)
    assert "log_var" in d, f"{path} has no log_var — not an NLL dump"
    LV, MM, PP, TT = d["log_var"], d["masks"], d["preds"], d["targets"]
    MI = d.get("masked_idx")
    Mw, K, Ns, _ = LV.shape
    has_mask = MI is not None and MI.shape[1] > 0
    subs = ("all", "msk", "vis") if has_mask else ("all",)
    s_var = {s: np.zeros((K, len(VARS))) for s in subs}   # sum sigma^2
    s_e2  = {s: np.zeros((K, len(VARS))) for s in subs}   # sum e^2
    s_cnt = {s: np.zeros((K, len(VARS))) for s in subs}
    for a in range(0, Mw, CHUNK):
        b = min(a + CHUNK, Mw)
        sig = np.exp(0.5 * LV[a:b].numpy().astype(np.float64)) * STD[None, None]
        err = (PP[a:b].numpy().astype(np.float64)
               - TT[a:b, :, :, :len(VARS)].numpy().astype(np.float64)) * STD[None, None]
        m = (MM[a:b, :, :, :len(VARS)].numpy() > 0.5) & KEEP[None, None]
        sel = None
        if has_mask:
            mi = MI[a:b].numpy()
            sel = np.zeros((b - a, Ns), bool)
            np.put_along_axis(sel, mi, True, axis=1)
        for s in subs:
            mm = (m if s == "all"
                  else m & sel[:, None, :, None] if s == "msk"
                  else m & ~sel[:, None, :, None])
            s_var[s] += (sig ** 2 * mm).sum(axis=(0, 2))
            s_e2[s]  += (err ** 2 * mm).sum(axis=(0, 2))
            s_cnt[s] += mm.sum(axis=(0, 2))
        del sig, err, m
    SIG[mr] = {"var": s_var, "e2": s_e2, "cnt": s_cnt, "subs": subs}
    print(f"  {NLL_RUN} {mr}: subsets {subs}, "
          f"{int(s_cnt['all'].sum()):,} valid predictions")
    del d, LV, MM, PP, TT

In [ ]:
# ── Predicted sigma vs lead time, per variable (RMSE overlaid) ───────────────
def _rms(sums, cnt):
    return np.sqrt(sums / np.maximum(cnt, 1))

STYLE = {("mr0.00", "all"): ("mr0.00 all stations",     "#1F5F6B"),
         ("mr0.50", "vis"): ("mr0.50 visible stations", "#2E7D8C"),
         ("mr0.50", "msk"): ("mr0.50 MASKED stations",  "#C4502A")}

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (vi, v) in zip(axes.ravel(), enumerate(VARS)):
    for (mr, s), (lab, col) in STYLE.items():
        if mr not in SIG or s not in SIG[mr]["subs"]:
            continue
        A = SIG[mr]
        sig = _rms(A["var"][s][:, vi], A["cnt"][s][:, vi])
        rms = _rms(A["e2"][s][:, vi],  A["cnt"][s][:, vi])
        ax.plot(range(NK), sig[KSL], "-", marker="o", ms=3, lw=1.6,
                color=col, label=f"sigma — {lab}")
        ax.plot(range(NK), rms[KSL], "--", lw=1.2, alpha=.75, color=col,
                label=f"RMSE — {lab}")
    ax.set_xticks(range(NK))
    ax.set_xticklabels(LEAD_F, rotation=45, ha="right", fontsize=7)
    ax.set_title(f"{v}  [{UNITS[v]}]"); ax.grid(alpha=.3)
axes.ravel()[0].set_ylabel("predicted sigma  /  realised RMSE")
lax = axes.ravel()[-1]; lax.axis("off")
h, l = axes.ravel()[0].get_legend_handles_labels()
lax.legend(h, l, loc="center", fontsize=7.5, frameon=False,
           title="solid = predicted sigma\ndashed = realised RMSE")
fig.suptitle("v30-nll — predicted uncertainty vs lead time, by variable", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Calibration table: realised RMSE / predicted sigma ──────────────────────
rows = {}
for (mr, s), (lab, _) in STYLE.items():
    if mr not in SIG or s not in SIG[mr]["subs"]:
        continue
    A = SIG[mr]
    for vi, v in enumerate(VARS):
        sig = _rms(A["var"][s][:, vi], A["cnt"][s][:, vi])
        rms = _rms(A["e2"][s][:, vi],  A["cnt"][s][:, vi])
        ratio = rms / np.where(sig > 0, sig, np.nan)
        rows[(lab, v)] = dict(zip(LEAD_F, ratio[KSL]))
cal = pd.DataFrame(rows).T
cal.index.names = ["subset", "variable"]
display(cal.style.format("{:.2f}")
        .background_gradient(cmap="RdBu_r", vmin=0.5, vmax=1.5)
        .set_caption("realised RMSE / predicted sigma — 1.0 calibrated, "
                     ">1 overconfident, <1 underconfident"))

grow = {}
for (mr, s), (lab, _) in STYLE.items():
    if mr not in SIG or s not in SIG[mr]["subs"]:
        continue
    A = SIG[mr]
    grow[lab] = {}
    for vi, v in enumerate(VARS):
        sig = _rms(A["var"][s][:, vi], A["cnt"][s][:, vi])
        grow[lab][v] = sig[-1] / sig[1] if sig[1] > 0 else np.nan
print("\ngrowth of predicted sigma, +6h relative to +30min "
      "(>1 = the model widens its own error bars with lead):")
display(pd.DataFrame(grow).T.round(2))

### Reading this section

**The requested plot** is the solid curve in each panel: predicted sigma against
lead time, one panel per variable, in physical units. Variance is sigma^2.

Three things to look for:

1. **Does sigma grow with lead?** It must — +6h is genuinely harder than
   +30min. The growth table quantifies it. A flat sigma would mean the variance
   head learned a per-variable constant and is not lead-aware.
2. **Does sigma track RMSE?** Solid and dashed curves of the same colour should
   lie on top of each other. Systematic separation is miscalibration and the
   table gives the factor. Mild overconfidence that worsens with lead is the
   common failure mode: predicting the *level* of error is easier than
   predicting its *growth*.
3. **Is sigma larger for masked stations?** The red pair (mr0.50 masked) should
   sit above the teal pair (mr0.50 visible). If it does, the model has learned
   that a station it cannot see is intrinsically less predictable — uncertainty
   responding to information content rather than only to variable and horizon,
   and the strongest evidence the NLL head is doing something useful.

The masked/visible split uses the **original evaluation masks** from
`masked_idx` in the dump, not a regenerated mask.